# Rail Corrugation
## Wavelength Feature Experiment

This notebook evaluates whether speed-normalised wavelength-domain features improve rail corrugation classification.

The baseline model achieves approximately 0.716 Macro F1 using vibration features.

Rather than using train speed directly, vibration frequencies are converted into estimated spatial wavelengths using:

λ = v / f

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.signal import welch

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier

DATA_DIR = Path("..")
TRAIN_PATH = DATA_DIR / "Train"

FS = 10000
WHEEL_DIAMETER = 0.85
TEETH = 90

In [2]:
labels = pd.read_csv(DATA_DIR / "Train_Labels.csv")
baseline = pd.read_csv(DATA_DIR / "rail_features.csv")

sample = pd.read_csv(TRAIN_PATH / "Train1.csv")

vib_cols = [c for c in sample.columns if "vibration" in c.lower()]

side1_vib = [c for c in vib_cols if any(f"position {p}" in c.lower() for p in [1, 3, 5, 7])]
side2_vib = [c for c in vib_cols if any(f"position {p}" in c.lower() for p in [2, 4, 6, 8])]

In [3]:
def estimate_speed(signal):
    transitions = np.sum(np.diff(signal) != 0)
    revolutions = (transitions / 2) / TEETH

    return revolutions * np.pi * WHEEL_DIAMETER

wavelength_bands = [
    (0.02, 0.05),
    (0.05, 0.10),
    (0.10, 0.20),
    (0.20, 0.40),
    (0.40, 0.80)
]

In [4]:
def wavelength_features(signal, speed):
    f, psd = welch(signal, fs=FS, nperseg=2048)

    f = f[1:]
    psd = psd[1:]
    wavelength = speed / f

    features = {}

    for low, high in wavelength_bands:
        mask = (wavelength >= low) & (wavelength < high)
        features[f"wl_{int(low*100)}_{int(high*100)}cm"] = psd[mask].sum()

    return features

In [5]:
rows = []

for _, row in labels.iterrows():
    data = pd.read_csv(TRAIN_PATH / row["filename"])

    speed = estimate_speed(data.iloc[:, 0].to_numpy())

    side1 = data[side1_vib].mean(axis=1).to_numpy()
    side2 = data[side2_vib].mean(axis=1).to_numpy()

    s1 = wavelength_features(side1, speed)
    s2 = wavelength_features(side2, speed)

    features = {}

    features.update({f"side1_{k}": v for k, v in s1.items()})
    features.update({f"side2_{k}": v for k, v in s2.items()})

    rows.append(features)

wl_df = pd.DataFrame(rows)

In [6]:
display(wl_df.head())

,side1_wl_2_5cm,side1_wl_5_10cm,side1_wl_10_20cm,side1_wl_20_40cm,side1_wl_40_80cm,side2_wl_2_5cm,side2_wl_5_10cm,side2_wl_10_20cm,side2_wl_20_40cm,side2_wl_40_80cm
0,0.000101,0.000013,3.352528e-07,8.266658e-08,0.000000,0.000087,0.000010,2.549564e-07,7.320711e-08,0.000000
1,0.000906,0.000289,1.636548e-03,1.245999e-03,0.000282,0.000996,0.000305,1.864019e-03,2.429667e-03,0.000340
2,0.000516,0.000139,2.248533e-04,6.458629e-04,0.000101,0.000611,0.000155,2.930916e-04,6.611945e-04,0.000076
3,0.000171,0.000034,4.828057e-04,5.482347e-04,0.000016,0.000220,0.000082,7.547326e-04,6.366263e-04,0.000037
4,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000


In [7]:
print("NaN:", wl_df.isna().sum().sum())
print("Infinite:", np.isinf(wl_df).sum().sum())

NaN: 0
Infinite: 0


In [8]:
wl_analysis = wl_df.copy()
wl_analysis["label"] = labels["label"]

display(
    wl_analysis.groupby("label").mean().T
)

label,Normal,Side I,Side II
side1_wl_2_5cm,0.000158,0.000420,0.000676
side1_wl_5_10cm,0.000077,0.000125,0.000223
side1_wl_10_20cm,0.000215,0.000623,0.000900
side1_wl_20_40cm,0.000263,0.001228,0.000885
side1_wl_40_80cm,0.000024,0.000079,0.000051
side2_wl_2_5cm,0.000159,0.000362,0.000656
side2_wl_5_10cm,0.000086,0.000108,0.000260
side2_wl_10_20cm,0.000239,0.000448,0.001503
side2_wl_20_40cm,0.000272,0.001018,0.001933
side2_wl_40_80cm,0.000029,0.000103,0.000058


In [9]:
X_base = baseline.drop(columns=["filename", "label"])
y = baseline["label"]

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rf = RandomForestClassifier(
    n_estimators=500,
    class_weight="balanced",
    random_state=42
)

In [10]:
base_scores = cross_val_score(
    rf,
    X_base,
    y,
    cv=cv,
    scoring="f1_macro"
)

print("Baseline:", base_scores.round(3))
print("Mean:", base_scores.mean().round(3))

Baseline: [0.725 0.652 0.779 0.756 0.666]
Mean: 0.716


In [11]:
wl_scores = cross_val_score(
    rf,
    wl_df,
    y,
    cv=cv,
    scoring="f1_macro"
)

print("Wavelength only:", wl_scores.round(3))
print("Mean:", wl_scores.mean().round(3))

Wavelength only: [0.587 0.563 0.709 0.589 0.708]
Mean: 0.631


In [12]:
X_combined = pd.concat(
    [X_base.reset_index(drop=True), wl_df],
    axis=1
)

combined_scores = cross_val_score(
    rf,
    X_combined,
    y,
    cv=cv,
    scoring="f1_macro"
)

print("Baseline + Wavelength:", combined_scores.round(3))
print("Mean:", combined_scores.mean().round(3))

Baseline + Wavelength: [0.642 0.652 0.815 0.73  0.685]
Mean: 0.705


In [13]:
results = pd.DataFrame({
    "Baseline": base_scores,
    "Wavelength": wl_scores,
    "Baseline + Wavelength": combined_scores
})

display(results)

,Baseline,Wavelength,Baseline + Wavelength
0,0.724755,0.587037,0.642105
1,0.652482,0.563321,0.652482
2,0.779487,0.709380,0.814976
3,0.755871,0.589474,0.729988
4,0.665633,0.707729,0.684547


In [14]:
display(
    pd.DataFrame({
        "Mean Macro F1": results.mean(),
        "Std": results.std()
    })
)

,Mean Macro F1,Std
Baseline,0.715646,0.055380
Wavelength,0.631388,0.071182
Baseline + Wavelength,0.704820,0.070447
